# Part 1 — Clean monthly PM2.5 (no hand merges)

This notebook is step zero for modeling.

**What we use:** the file `df_model_monthly.csv` that Lena already built. We are **not** re-joining OpenAQ, weather, or land cover here — that work is already inside this CSV.

**The one thing I fix:** some rows used **annual_fill** for PM2.5 (same yearly number copied into every month). That could cause data leakage, make fake seasonality and can make models look better than they are. We **drop** those rows.

**What we keep:** only `openaq_monthly` — real measured PM2.5 for that month.

**Output:** `datasets/part1_openaq_base.csv` for Part2.

**Heads-up on names:** we set `openaq_id` = `eoi_code`. That is the **EEA station code** (e.g. IT1975A), not OpenAQ’s internal numeric id. Part2 needs a stable station id per row; this is it.


### Imports

Standard stuff: pandas for tables, pathlib for paths.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


Libraries loaded


### Paths

`MERGED_MONTHLY` is the big pre-merged table. `OUTPUT` is what we hand to Part2. Keeping paths at the top means we only edit one place if files move.


In [2]:
ROOT = Path(".")
MERGED_MONTHLY = ROOT / "df_model_monthly.csv"
OUTPUT = ROOT / "datasets" / "part1_openaq_base.csv"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("Merged input:", MERGED_MONTHLY)
print("Output:", OUTPUT)


Merged input: df_model_monthly.csv
Output: datasets/part1_openaq_base.csv


### Load the merged table

Each row = one station (`eoi_code`) × one month. Peek at `data_source`: that column is the whole reason this notebook exists — we’ll filter on it next.


In [3]:
df = pd.read_csv(MERGED_MONTHLY)
print("Shape (all rows):", df.shape)
print("Columns:", list(df.columns))
display(df.head())


Shape (all rows): (5759, 20)
Columns: ['eoi_code', 'Year', 'Month', 'Season', 'data_source', 'PM2_5', 'PM10', 'NO2', 'O3', 'Altitude', 'Latitude', 'Longitude', 'Station_Type', 'Station_Area', 'City', 'Green_Ratio', 'Population_Density', 'Temp_Mean', 'Wind_Speed', 'Precipitation']


,eoi_code,Year,Month,Season,data_source,PM2_5,PM10,NO2,O3,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density,Temp_Mean,Wind_Speed,Precipitation
0,IT1975A,2020,1,Winter,annual_fill,16.00,38.3,26.1,26.6,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,3.845161,6.132258,31.3
1,IT1975A,2020,2,Winter,openaq_monthly,25.70,29.3,27.8,47.3,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,7.737931,8.710345,6.5
2,IT1975A,2020,3,Spring,openaq_monthly,15.30,19.7,NaN,71.5,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,8.509677,8.977419,90.6
3,IT1975A,2020,4,Spring,openaq_monthly,8.17,12.3,NaN,60.8,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,14.156667,7.590000,22.6
4,IT2250A,2020,1,Winter,openaq_monthly,8.50,17.9,13.7,NaN,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0,9.087097,9.038710,38.4


### Count rows by `data_source`

Quick sanity check: you should see both `openaq_monthly` and `annual_fill`. Note the percentages — we expect to lose ~30% of rows when we drop annual fill, but the target becomes trustworthy.


In [4]:
if "data_source" not in df.columns:
    raise ValueError("Expected column data_source. Use df_model_monthly.csv from this project.")

counts = df["data_source"].value_counts()
print(counts)
n = len(df)
print()
print("Real OpenAQ monthly %:", round(counts.get("openaq_monthly", 0) / n * 100, 2))
print("Annual fill %:", round(counts.get("annual_fill", 0) / n * 100, 2))


data_source
openaq_monthly    3986
annual_fill       1773
Name: count, dtype: int64

Real OpenAQ monthly %: 69.21
Annual fill %: 30.79


### Drop annual fill

After this, **every PM2_5 value is a real monthly observation**. We also drop `data_source` because it's always the same now.

If this cell makes the dataframe tiny, something's wrong with the input file.


In [5]:
df = df[df["data_source"] == "openaq_monthly"].copy()
df = df.drop(columns=["data_source"])

print("Shape after filter:", df.shape)
display(df.head())


Shape after filter: (3986, 19)


,eoi_code,Year,Month,Season,PM2_5,PM10,NO2,O3,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density,Temp_Mean,Wind_Speed,Precipitation
1,IT1975A,2020,2,Winter,25.70,29.3,27.8,47.3,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,7.737931,8.710345,6.5
2,IT1975A,2020,3,Spring,15.30,19.7,NaN,71.5,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,8.509677,8.977419,90.6
3,IT1975A,2020,4,Spring,8.17,12.3,NaN,60.8,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0,14.156667,7.590000,22.6
4,IT2250A,2020,1,Winter,8.50,17.9,13.7,NaN,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0,9.087097,9.038710,38.4
5,IT2250A,2020,2,Winter,9.68,19.5,11.2,NaN,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0,10.903448,11.848276,12.2


### Build `date` and `openaq_id`

`date` = first day of that month (easier for sorting and plots). `openaq_id` duplicates `eoi_code` so Part2’s code can group by `openaq_id` without us rewriting all of Part2.


In [6]:
df["date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))

if "eoi_code" not in df.columns:
    raise ValueError("Expected column eoi_code.")

df["openaq_id"] = df["eoi_code"].astype(str)

display(df[["openaq_id", "eoi_code", "Year", "Month", "date", "PM2_5"]].head())


,openaq_id,eoi_code,Year,Month,date,PM2_5
1,IT1975A,IT1975A,2020,2,2020-02-01,25.70
2,IT1975A,IT1975A,2020,3,2020-03-01,15.30
3,IT1975A,IT1975A,2020,4,2020-04-01,8.17
4,IT2250A,IT2250A,2020,1,2020-01-01,8.50
5,IT2250A,IT2250A,2020,2,2020-02-01,9.68


### Column order

Just housekeeping: ids and target up front so the CSV is readable when someone opens it in Excel.


In [7]:
front = [
    "openaq_id", "eoi_code", "Year", "Month", "date", "Season",
    "PM2_5", "PM10", "NO2", "O3",
    "Temp_Mean", "Wind_Speed", "Precipitation",
    "Altitude", "Latitude", "Longitude",
    "Station_Type", "Station_Area", "City",
    "Green_Ratio", "Population_Density",
]
cols = [c for c in front if c in df.columns]
rest = [c for c in df.columns if c not in cols]
df = df[cols + rest].copy()
print("Columns:", list(df.columns))


Columns: ['openaq_id', 'eoi_code', 'Year', 'Month', 'date', 'Season', 'PM2_5', 'PM10', 'NO2', 'O3', 'Temp_Mean', 'Wind_Speed', 'Precipitation', 'Altitude', 'Latitude', 'Longitude', 'Station_Type', 'Station_Area', 'City', 'Green_Ratio', 'Population_Density']


### Quality checks

Things we look for: no missing PM2.5, no duplicate station-month keys, and a sensible date range. Missing **other** columns (O3, PM10) is ok for now — Part2/3 can impute or drop features; the target must be solid.


In [8]:
key_cols = ["openaq_id", "Year", "Month"]
print("Rows:", len(df))
print("Missing PM2_5:", int(df["PM2_5"].isna().sum()))
print("Duplicate keys:", int(df.duplicated(subset=key_cols).sum()))
print("Date range:", df["date"].min(), "to", df["date"].max())
print()
print("Missing % (top 15):")
print((df.isna().mean() * 100).sort_values(ascending=False).head(15).round(2))


Rows: 3986
Missing PM2_5: 0
Duplicate keys: 0
Date range: 2020-01-01 00:00:00 to 2025-11-01 00:00:00

Missing % (top 15):
O3                    46.64
PM10                  15.15
Population_Density     2.41
NO2                    2.28
Altitude               0.08
Precipitation          0.00
Green_Ratio            0.00
City                   0.00
Station_Area           0.00
Station_Type           0.00
Longitude              0.00
Latitude               0.00
openaq_id              0.00
Wind_Speed             0.00
eoi_code               0.00
dtype: float64


### Save

Write `part1_openaq_base.csv`. everyone should run this once (or after changing `df_model_monthly.csv`) before Part2.


In [9]:
df.to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT, "| shape:", df.shape)
display(df.head())


Saved: datasets/part1_openaq_base.csv | shape: (3986, 21)


,openaq_id,eoi_code,Year,Month,date,Season,PM2_5,PM10,NO2,O3,Temp_Mean,Wind_Speed,Precipitation,Altitude,Latitude,Longitude,Station_Type,Station_Area,City,Green_Ratio,Population_Density
1,IT1975A,IT1975A,2020,2,2020-02-01,Winter,25.70,29.3,27.8,47.3,7.737931,8.710345,6.5,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
2,IT1975A,IT1975A,2020,3,2020-03-01,Spring,15.30,19.7,NaN,71.5,8.509677,8.977419,90.6,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
3,IT1975A,IT1975A,2020,4,2020-04-01,Spring,8.17,12.3,NaN,60.8,14.156667,7.590000,22.6,61.0,45.0383,9.6692,Background,Urban,Piacenza,0.717906,868.0
4,IT2250A,IT2250A,2020,1,2020-01-01,Winter,8.50,17.9,13.7,NaN,9.087097,9.038710,38.4,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0
5,IT2250A,IT2250A,2020,2,2020-02-01,Winter,9.68,19.5,11.2,NaN,10.903448,11.848276,12.2,3.0,42.1022,11.7844,Traffic,Urban,Civitavecchia,0.811256,699.0


### Next step

Open **`part2_features_and_validation.ipynb`** and point it at `datasets/part1_openaq_base.csv`.
